## Alteracoes para acelerar o treino e a inferencia

- **Resolucao**: `img_height`/`img_width` reduzidos de 61 para 32 (a tarefa e classificar a cor de fundo predominante, nao reconhecer a letra -- o fundo ocupa ~83% dos pixels em uma amostra real).
- **`interpolation="area"`**: reamostragem mais adequada para reduzir imagens (evita aliasing); o mesmo metodo foi usado em `compvision.py` para manter treino e inferencia consistentes.
- **`Flatten()` -> `GlobalAveragePooling2D()`**: elimina a camada `Dense` gigante que vinha do flatten (a maior parte dos parametros do modelo) sem perder informacao relevante para classificar cor.
- **`EarlyStopping`**: interrompe o treino quando a acuracia de validacao para de melhorar, em vez de sempre rodar todas as epocas.

**Depois de rodar este notebook, `modelo.keras` tera uma arquitetura nova -- e preciso treinar de novo antes de usar com `compvision.py`** (que tambem foi atualizado para esperar entradas 32x32).

## 1. Configuração inicial

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import PIL
import tensorflow as tf
import pathlib

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

## 2. Dados

As imagens de treino ficam em `data/images/`, organizadas em três subpastas -- uma por classe (`amarelo`, `preto`, `verde`), os três resultados possíveis de um quadrado do tabuleiro. Cada imagem mostra uma letra em branco sobre o fundo daquela cor (veja `a.png` como exemplo).

In [ ]:
data_dir = pathlib.Path("../data/images/")

In [ ]:
image_count = len(list(data_dir.glob('*/*.png')))
print('Total de imagens:', image_count)

# Contagem por classe -- se uma pasta tiver muito mais imagens que as
# outras (desbalanceamento), ou se o total for muito pequeno, o
# modelo pode aprender a so prever a classe majoritaria em vez de
# realmente distinguir as cores.
for pasta in sorted(p for p in data_dir.iterdir() if p.is_dir()):
    n = len(list(pasta.glob('*.png')))
    print(f'  {pasta.name}: {n} imagens')

In [ ]:
# Conferencia visual: abre uma imagem da classe "amarelo" para checar
# se o dataset foi montado corretamente (letra branca sobre fundo na
# cor esperada).
yellow = list(data_dir.glob('amarelo/*'))
PIL.Image.open(str(yellow[0]))

## 3. Hiperparâmetros e divisão treino/validação

`image_dataset_from_directory` faz a divisão 80/20 (treino/validação) e já devolve os dados em lotes (`batch_size`), redimensionados para `(img_height, img_width)`.

In [ ]:
# batch_size pequeno de proposito: com poucas imagens por classe
# (letras x 3 cores costuma dar um dataset pequeno), um batch grande
# significa POUCAS atualizacoes de peso por epoca -- o modelo pode
# parecer 'nao aprender' so porque quase nao deu passos de gradiente,
# nao porque a tarefa seja dificil. Ajuste para cima com cuidado e so
# se a contagem de imagens (celula acima) mostrar um dataset grande.
batch_size = 8
# Resolucao reduzida (61 -> 32): a tarefa e classificar a COR de fundo
# do quadrado (amarelo/preto/verde), nao reconhecer a letra. Na imagem
# de amostra enviada, o fundo ja ocupa ~83% dos pixels, entao uma
# resolucao bem menor preserva o sinal relevante e acelera tanto o
# treino (menos pixels por convolucao) quanto a inferencia. Se a
# acuracia cair ao testar, aumente de volta (ex.: 48) e retreine.
img_height = 32
img_width = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  interpolation="area",
  batch_size=batch_size)

In [ ]:
val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  interpolation="area",
  batch_size=batch_size)

In [ ]:
# image_dataset_from_directory usa o nome das subpastas como classes,
# em ordem alfabetica -- por isso "amarelo" (indice 0), "preto"
# (indice 1) e "verde" (indice 2). predict_square(), em compvision.py,
# depende dessa ordem especifica para converter a previsao do modelo
# de volta para o valor do jogo (0/1/2).
class_names = train_ds.class_names
print(class_names)

### Conferindo visualmente um lote de treino

Mostra até 9 imagens de um único lote com seus rótulos, para confirmar visualmente que os dados e as classes estão corretos antes de treinar qualquer coisa.

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  # min(9, ...) em vez de 9 fixo: com batch_size pequeno (agora 8), o
  # lote pode ter menos de 9 imagens.
  n_exemplos = min(9, images.shape[0])
  for i in range(n_exemplos):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")

In [ ]:
# Confere o formato de um lote: deve ser
# (batch_size, img_height, img_width, 3) para as imagens e
# (batch_size,) para os rotulos.
for image_batch, labels_batch in train_ds:
  print(image_batch.shape)
  print(labels_batch.shape)
  break

In [ ]:
# cache(): mantem as imagens (ja decodificadas) em memoria depois da
# primeira epoca, evitando reler e redecodificar os arquivos do disco
# a cada epoca. shuffle(): embaralha a ordem dos lotes a cada epoca.
# prefetch(): prepara o proximo lote enquanto o atual ainda esta
# sendo processado pelo modelo, sobrepondo I/O e computo.
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

### Nota: conferência de normalização (não usada pelo modelo)

As duas células abaixo só existem para conferir visualmente que `Rescaling` funciona como esperado (pixels em `[0, 255]` viram `[0, 1]`). O modelo definido mais adiante tem sua **própria** camada `Rescaling` embutida -- essa instância aqui não é reusada por ele.

In [ ]:
normalization_layer = layers.Rescaling(1./255)

In [ ]:
normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
image_batch, labels_batch = next(iter(normalized_ds))
first_image = image_batch[0]
# Notice the pixel values are now in `[0,1]`.
print(np.min(first_image), np.max(first_image))

## 4. Definição do modelo

Uma CNN pequena de propósito, pensada para a tarefa real (classificar a cor de fundo predominante de um quadrado, não reconhecer a letra desenhada nele) e para o tamanho do dataset disponível. As decisões de arquitetura estão comentadas na célula abaixo; resumo rápido:

| Camada | Papel |
|---|---|
| `Rescaling` | Normaliza os pixels de `[0, 255]` para `[0, 1]` |
| `RandomRotation`/`RandomTranslation`/`RandomZoom` | Data augmentation -- só ativas durante `fit()`, viram no-op na inferência |
| `Conv2D` + `MaxPooling2D` | Extrai características espaciais básicas (bordas, regiões de cor) |
| `GlobalAveragePooling2D` | Reduz cada mapa de ativação a uma média por canal, no lugar de um `Flatten()` |
| `Dense(16)` + `Dense(num_classes)` | Classificador final sobre as características extraídas |

In [ ]:
num_classes = len(class_names)

# Flatten() -> GlobalAveragePooling2D(): o Flatten() gerava um vetor
# gigante (H x W x canais) alimentando a camada Dense seguinte, que
# concentrava a maior parte dos parametros (e do tempo de computacao)
# do modelo inteiro. GlobalAveragePooling2D reduz cada mapa de ativacao
# a um unico valor (a media) -- o que faz sentido aqui, ja que estamos
# classificando uma cor de fundo predominante, nao uma forma -- e
# derruba drasticamente o numero de parametros da camada Dense sem
# perder a informacao relevante para essa tarefa.
#
# RandomRotation/RandomTranslation/RandomZoom: com so ~26 imagens por
# classe (78 no total), o modelo ve os MESMOS exemplos em toda epoca,
# o que favorece decorar em vez de generalizar. Essas camadas aplicam
# pequenas variacoes aleatorias (so durante o treino -- na inferencia
# via predict()/tf.function elas nao fazem nada) para o modelo ver
# cada imagem de um jeito um pouco diferente a cada epoca. Os valores
# sao propositalmente pequenos para nao alterar muito a proporcao da
# cor de fundo, que e o que importa para essa tarefa.
model = Sequential([
  layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
  layers.RandomRotation(0.05),
  layers.RandomTranslation(0.08, 0.08),
  layers.RandomZoom(0.08),
  layers.Conv2D(8, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.GlobalAveragePooling2D(),
  layers.Dense(16, activation='relu'),
  layers.Dense(num_classes)
])

In [ ]:
# from_logits=True: a ultima camada do modelo (Dense(num_classes)) nao
# tem ativacao softmax -- ela devolve logits "crus", e a propria loss
# aplica o softmax internamente de forma numericamente mais estavel
# do que fazer isso em duas etapas separadas.
model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [ ]:
# Confere a arquitetura e a contagem de parametros por camada -- e
# onde da pra ver concretamente o efeito de GlobalAveragePooling2D
# no tamanho da camada Dense seguinte (veja a celula de definicao do
# modelo, acima).
model.summary()

## 5. Treinamento

Treina o modelo com `EarlyStopping` monitorando `val_loss` (interrompe e restaura os melhores pesos se o loss de validação parar de melhorar por `patience` épocas seguidas), em vez de rodar sempre um número fixo de épocas.

In [ ]:
import time

epochs = 40
# monitor="val_loss" em vez de "val_accuracy": com um conjunto de
# validacao pequeno (~15 imagens), a acuracia so pode assumir poucos
# valores possiveis (multiplos de 1/15) e fica "presa" nesses saltos
# por varias epocas mesmo com o modelo melhorando de verdade por
# baixo. val_loss e continuo e reflete a evolucao real do treino de
# forma mais confiavel nesse cenario.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=8,
    restore_best_weights=True,
)

inicio = time.time()
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs,
  callbacks=[early_stopping]
)
print(f"Tempo total de treino: {time.time() - inicio:.1f}s ({len(history.epoch)} epocas)")
print("accuracy por epoca:", [round(a, 3) for a in history.history['accuracy']])
print("val_accuracy por epoca:", [round(a, 3) for a in history.history['val_accuracy']])
print("val_loss por epoca:", [round(a, 3) for a in history.history['val_loss']])

## 6. Avaliação: curvas de treino

Plota a acurácia e o loss de treino/validação por época. Vale observar os dois juntos: se a acurácia de validação "travar" em um valor por várias épocas mas o loss continuar caindo, é sinal de que o conjunto de validação é pequeno demais para essa métrica variar de forma contínua (ela só consegue assumir múltiplos de 1/tamanho_da_validação) -- o loss, sendo contínuo, continua refletindo o progresso real do modelo nesse cenário.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

# range(epochs) supunha que o treino sempre rodaria todas as epocas do
# teto (epochs=30). Com EarlyStopping o treino pode parar antes disso,
# entao o eixo x precisa refletir quantas epocas rodaram de fato.
epochs_range = range(len(acc))

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

## 7. Teste rápido e exportação do modelo

Faz uma previsão de sanidade com uma imagem conhecida (a amostra "amarelo") e salva o modelo treinado em `modelo.keras` -- o arquivo que `compvision.py` carrega para a estratégia de deep learning. `target_size` usa as mesmas variáveis `img_height`/`img_width` definidas lá em cima, para garantir que a imagem de teste passe pelo mesmo pré-processamento usado no treino.

In [ ]:
caminho_imagem = pathlib.Path("../data/images/amarelo/a.png")
img = tf.keras.utils.load_img(caminho_imagem, target_size=(img_height, img_width))
img_array = tf.keras.utils.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

previsoes = model.predict(img_array)
print(previsoes)
model.save("modelo.keras")